# IIIXR validation 3D | sanity

Enable Internet. The notebook fetches `REPO_REVISION` from GitHub and downloads the pinned validation clips from Hugging Face. Fresh mode also downloads the required models and uses GPU T4 x2. Kaggle Inputs can stay empty.

The default mode is `replay`. Download the resulting ZIP and open `run/index.html` after extraction.


In [ ]:
SCOPE = "sanity"
MODE = "replay"  # replay uses CPU; fresh uses GPU T4 x2
REPO_URL = "https://github.com/leetranwyng/quantiphy_trannl.git"
REPO_REVISION = "main"  # branch, tag or commit containing the IIIXR 3D module
GITHUB_SECRET = ""  # Kaggle Secret name for a GitHub token, only for a private repo
SHARDS = 1
SHARD = 0
MOLMO = "disabled"  # vllm-bnb requires a compatible vLLM runtime
RESUME = ""  # optional checkpoint path for fresh
INSTALL = True
OUTPUT = "/kaggle/working/iiixr_3d_sanity"
CACHE = "/kaggle/temp/iiixr_3d"


In [ ]:
import base64
import os
import subprocess
import sys
import tempfile
from pathlib import Path

def checkout():
    cache = Path(CACHE).resolve()
    cache.mkdir(parents=True, exist_ok=True)
    target = Path(tempfile.mkdtemp(prefix="checkout-", dir=cache))
    git = ["git", "-C", str(target)]
    subprocess.run([*git, "init", "--quiet"], check=True)
    auth = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if GITHUB_SECRET:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret(GITHUB_SECRET)
        header = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        index = int(auth.get("GIT_CONFIG_COUNT", "0"))
        auth[f"GIT_CONFIG_KEY_{index}"] = "http.https://github.com/.extraHeader"
        auth[f"GIT_CONFIG_VALUE_{index}"] = f"Authorization: Basic {header}"
        auth["GIT_CONFIG_COUNT"] = str(index + 1)
    try:
        subprocess.run([*git, "fetch", "--depth=1", "--no-tags", REPO_URL, REPO_REVISION],
                       env=auth, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Git fetch failed. Check Kaggle Internet, REPO_URL and REPO_REVISION. "
            "Set GITHUB_SECRET to the Kaggle Secret name if the repository is private."
        ) from exc
    subprocess.run([*git, "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)
    print("Git checkout", subprocess.check_output([*git, "rev-parse", "HEAD"], text=True).strip())
    return target

source = checkout()
driver = source / "solutions/iiixr/kaggle/three_d/workflow.py"
if not driver.is_file():
    raise RuntimeError(f"{REPO_REVISION} is missing the IIIXR 3D module. Push the 3D files to GitHub first.")
if INSTALL:
    if MODE == "fresh":
        deps = ["-r", str(driver.with_name("requirements.txt"))]
    else:
        deps = ["numpy>=1.26,<3", "pandas>=2.1,<3", "scipy>=1.13",
                "opencv-python-headless>=4.10", "huggingface-hub>=0.34,<2"]
    subprocess.run([sys.executable, "-m", "pip", "install", *deps], check=True)


In [ ]:
import json

config = {key: globals()[key] for key in ("SCOPE", "MODE", "OUTPUT", "CACHE", "SHARDS", "SHARD", "MOLMO", "RESUME")}
subprocess.run(
    [sys.executable, "-c",
     "import json,sys; from pathlib import Path; from solutions.iiixr.kaggle.three_d.notebook import launch; launch(Path.cwd(), json.loads(sys.argv[1]))",
     json.dumps(config)],
    cwd=source, check=True,
)
